# MinHash Deep Dive: Choosing the Right Encoding Method

This notebook explores the different MinHash encoding methods and when to use each one.

## Key Concepts

- **MinHash** compresses data into fixed-size signatures that preserve Jaccard similarity
- **Binary methods** use permutation-based hashing (fast, for fingerprints)
- **String methods** use SHA-1 hashing (for text/tokens)
- **Binary and string signatures are NOT comparable** (different hash functions!)

In [ ]:
import time

import numpy as np

from tmap import MinHash

print("Ready for MinHash exploration!")


Ready for MinHash exploration!


## Method Overview

| Method | Input Type | Best For | Speed |
|--------|-----------|----------|-------|
| `batch_from_binary_array` | 2D numpy array | **Fingerprints** | Fastest |
| `from_binary_array` | 1D numpy array | Single fingerprint | Fast |
| `batch_from_sparse_binary_array` | List of index lists | Sparse fingerprints | Fast |
| `from_sparse_binary_array` | List of indices | Single sparse | Fast |
| `batch_from_string_array` | List of string lists | **Text/tokens** | Slower |
| `from_string_array` | List of strings | Single text | Slower |

## 1. Binary Methods (for Fingerprints)

Use these when your data is binary (0/1) - molecular fingerprints, one-hot encodings, etc.

In [ ]:
# Create sample binary fingerprints
np.random.seed(42)
n_samples = 1000
n_bits = 2048

# Dense format: (n_samples, n_bits) array of 0s and 1s
fingerprints_dense = (np.random.rand(n_samples, n_bits) < 0.1).astype(np.uint8)

print(f"Dense shape: {fingerprints_dense.shape}")
print(f"Average bits set: {fingerprints_dense.sum(axis=1).mean():.1f}")
print(f"Memory: {fingerprints_dense.nbytes / 1024:.1f} KB")


Dense shape: (1000, 2048)
Average bits set: 204.6
Memory: 2000.0 KB


In [ ]:
# RECOMMENDED: batch_from_binary_array for dense fingerprints
mh = MinHash(num_perm=128, seed=42)

start = time.time()
signatures = mh.batch_from_binary_array(fingerprints_dense)
elapsed = time.time() - start

print("batch_from_binary_array:")
print(f"  Time: {elapsed*1000:.1f} ms")
print(f"  Output shape: {signatures.shape}")
print(f"  Output dtype: {signatures.dtype}")


batch_from_binary_array:
  Time: 100.8 ms
  Output shape: (1000, 128)
  Output dtype: uint64


In [ ]:
# Alternative: from_binary_array for single fingerprint
single_fp = fingerprints_dense[0]

start = time.time()
single_sig = mh.from_binary_array(single_fp)
elapsed = time.time() - start

print("from_binary_array (single):")
print(f"  Time: {elapsed*1000:.2f} ms")
print(f"  Output shape: {single_sig.shape}")


from_binary_array (single):
  Time: 0.45 ms
  Output shape: (128,)


## 2. Sparse Methods (for Large/Sparse Fingerprints)

When fingerprints are very sparse (few bits set), storing only the indices is more efficient.

In [ ]:
# Convert dense to sparse (list of active indices per sample)
fingerprints_sparse = [np.where(fp == 1)[0].tolist() for fp in fingerprints_dense]

print(f"Sparse format: list of {len(fingerprints_sparse)} index lists")
print(f"Example indices[0]: {fingerprints_sparse[0][:10]}... ({len(fingerprints_sparse[0])} total)")


Sparse format: list of 1000 index lists
Example indices[0]: [6, 10, 29, 32, 37, 42, 56, 58, 68, 72]... (222 total)


In [ ]:
# batch_from_sparse_binary_array
start = time.time()
signatures_sparse = mh.batch_from_sparse_binary_array(fingerprints_sparse)
elapsed = time.time() - start

print("batch_from_sparse_binary_array:")
print(f"  Time: {elapsed*1000:.1f} ms")
print(f"  Output shape: {signatures_sparse.shape}")

# Verify: sparse and dense produce same signatures
print(f"\nSparse == Dense signatures: {np.allclose(signatures, signatures_sparse)}")


batch_from_sparse_binary_array:
  Time: 6.8 ms
  Output shape: (1000, 128)

Sparse == Dense signatures: True


## 3. String Methods (for Text/Tokens)

Use these when your data is text or categorical tokens.

In [ ]:
# Sample text data (e.g., tokenized documents)
documents = [
    ["hello", "world", "python", "programming"],
    ["hello", "python", "data", "science"],
    ["machine", "learning", "python", "data"],
    ["deep", "learning", "neural", "network"],
    ["hello", "world", "greeting", "message"],
]

print("Sample documents:")
for i, doc in enumerate(documents):
    print(f"  {i}: {doc}")


Sample documents:
  0: ['hello', 'world', 'python', 'programming']
  1: ['hello', 'python', 'data', 'science']
  2: ['machine', 'learning', 'python', 'data']
  3: ['deep', 'learning', 'neural', 'network']
  4: ['hello', 'world', 'greeting', 'message']


In [ ]:
# batch_from_string_array for text
mh_string = MinHash(num_perm=128, seed=42)

start = time.time()
signatures_text = mh_string.batch_from_string_array(documents)
elapsed = time.time() - start

print("batch_from_string_array:")
print(f"  Time: {elapsed*1000:.1f} ms")
print(f"  Output shape: {signatures_text.shape}")


batch_from_string_array:
  Time: 2.3 ms
  Output shape: (5, 128)


In [ ]:
# Check similarities between documents
from tmap import LSHForest

lsh = LSHForest(d=128, l=32)
lsh.batch_add(signatures_text)
lsh.index()

# Query: which documents are most similar to doc 0?
neighbors = lsh.query(signatures_text[0], k=3)
print(f"\nDocuments most similar to doc 0 ({documents[0]}):")
for idx in neighbors:
    print(f"  Doc {idx}: {documents[idx]}")



Documents most similar to doc 0 (['hello', 'world', 'python', 'programming']):
  Doc 0: ['hello', 'world', 'python', 'programming']
  Doc 1: ['hello', 'python', 'data', 'science']


## 4. Critical Warning: Binary vs String Incompatibility

**Binary and string methods use different hash functions!**

- Binary: Permutation-based (fast, Numba-accelerated)
- String: SHA-1 based (slower, handles arbitrary strings)

**You CANNOT compare signatures from different methods!**

In [ ]:
# Demonstration: same data, different methods = different signatures

# Create identical data in both formats
binary_fp = np.array([1, 0, 1, 1, 0, 0, 1, 0], dtype=np.uint8)
string_set = ["0", "2", "3", "6"]  # Indices where bits are 1

mh = MinHash(num_perm=32, seed=42)

sig_binary = mh.from_binary_array(binary_fp)
sig_string = mh.from_string_array(string_set)

print("Same data, different methods:")
print(f"  Binary signature: {sig_binary[:8]}...")
print(f"  String signature: {sig_string[:8]}...")
print(f"\n  Are they equal? {np.array_equal(sig_binary, sig_string)}")
print("\n⚠️  NEVER mix binary and string signatures in the same LSHForest!")


Same data, different methods:
  Binary signature: [  56972561  183869622  526864543  755772474   68574553  991681409
  685147122 1506162343]...
  String signature: [ 931091212  738920804   67677908  726905798 1095777755  371074316
  409203187  116074172]...

  Are they equal? False

⚠️  NEVER mix binary and string signatures in the same LSHForest!


## 5. Performance Comparison

In [ ]:
# Benchmark different methods
n_samples = 5000
n_bits = 2048

# Prepare data
np.random.seed(42)
fp_dense = (np.random.rand(n_samples, n_bits) < 0.05).astype(np.uint8)  # 5% density
fp_sparse = [np.where(fp == 1)[0].tolist() for fp in fp_dense]

mh = MinHash(num_perm=128, seed=42)

# Benchmark batch_from_binary_array
start = time.time()
_ = mh.batch_from_binary_array(fp_dense)
time_dense = time.time() - start

# Benchmark batch_from_sparse_binary_array
start = time.time()
_ = mh.batch_from_sparse_binary_array(fp_sparse)
time_sparse = time.time() - start

print(f"Performance comparison ({n_samples} samples, {n_bits} bits, 5% density):")
print(f"  batch_from_binary_array:        {time_dense*1000:6.1f} ms")
print(f"  batch_from_sparse_binary_array: {time_sparse*1000:6.1f} ms")
print("\nRecommendation: Use dense for small/medium datasets, sparse for very large/sparse data")


Performance comparison (5000 samples, 2048 bits, 5% density):
  batch_from_binary_array:          24.8 ms
  batch_from_sparse_binary_array:   17.7 ms

Recommendation: Use dense for small/medium datasets, sparse for very large/sparse data


## 6. Choosing num_perm

The `num_perm` parameter controls the signature size:

| num_perm | Accuracy | Speed | Memory | Use Case |
|----------|----------|-------|--------|----------|
| 64 | Lower | Fastest | Lowest | Quick exploration |
| 128 | Good | Fast | Low | **Default, most cases** |
| 256 | High | Medium | Medium | High precision needed |
| 512 | Very High | Slower | Higher | Very similar items |

In [ ]:
# Demonstrate num_perm effect on similarity accuracy
np.random.seed(42)

# Create two fingerprints with known Jaccard similarity
fp1 = np.zeros(1000, dtype=np.uint8)
fp2 = np.zeros(1000, dtype=np.uint8)

# Set bits to achieve ~0.7 Jaccard similarity
fp1[:100] = 1  # 100 bits in fp1
fp2[:70] = 1   # 70 shared + 30 unique to fp2
fp2[100:130] = 1

# True Jaccard
intersection = np.sum(fp1 & fp2)
union = np.sum(fp1 | fp2)
true_jaccard = intersection / union
print(f"True Jaccard similarity: {true_jaccard:.4f}")

# Estimate Jaccard with different num_perm values
print("\nEstimated Jaccard by num_perm:")
for num_perm in [32, 64, 128, 256, 512]:
    mh = MinHash(num_perm=num_perm, seed=42)
    sig1 = mh.from_binary_array(fp1)
    sig2 = mh.from_binary_array(fp2)

    # Estimate Jaccard as fraction of matching hash values
    estimated_jaccard = np.mean(sig1 == sig2)
    error = abs(estimated_jaccard - true_jaccard)
    print(f"  num_perm={num_perm:3d}: {estimated_jaccard:.4f} (error: {error:.4f})")


True Jaccard similarity: 0.5385

Estimated Jaccard by num_perm:
  num_perm= 32: 0.5938 (error: 0.0553)
  num_perm= 64: 0.5156 (error: 0.0228)
  num_perm=128: 0.5391 (error: 0.0006)
  num_perm=256: 0.5391 (error: 0.0006)
  num_perm=512: 0.5098 (error: 0.0287)


## Summary: Decision Tree

```
What is your data?
│
├─► Binary fingerprints (0/1 arrays)
│   │
│   ├─► Dense format (numpy 2D array)
│   │   └─► batch_from_binary_array() ✓ RECOMMENDED
│   │
│   └─► Sparse format (list of index lists)
│       └─► batch_from_sparse_binary_array()
│
└─► Text/tokens (strings)
    └─► batch_from_string_array()

⚠️ Never mix binary and string signatures!
```